In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# ============================================================
# PAPER 1
# Swin Transformer Molecular Subtype Classification
# Version 1
# ============================================================

print("=" * 80)
print("Paper 1 : Swin Transformer Breast Cancer Molecular Subtype Classification")
print("=" * 80)
 #============================================================
# STANDARD LIBRARIES
# ============================================================

import os
import gc
import time
import copy
import random
import warnings
from pathlib import Path

# ============================================================
# SCIENTIFIC LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

# ============================================================
# PYTORCH
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

from torchvision import datasets
from torchvision import transforms
from torchvision.models import swin_t
from torchvision.models import Swin_T_Weights
# ============================================================
# SCIKIT-LEARN
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    balanced_accuracy_score
)
# ============================================================
# MIXED PRECISION
# ============================================================

from torch.cuda.amp import (
    autocast,
    GradScaler
)
# ============================================================
# WARNINGS
# ============================================================

warnings.filterwarnings("ignore")

print("\nLibraries imported successfully.")
# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

IMAGE_SIZE = 224

NUM_CLASSES = 4

BATCH_SIZE = 64

NUM_WORKERS = min(4, os.cpu_count())

EPOCHS = 35

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.1

EARLY_STOPPING_PATIENCE = 8

GRADIENT_CLIP = 1.0

USE_AMP = True

PIN_MEMORY = True

PERSISTENT_WORKERS = True

MODEL_NAME = "Swin-T"



# ============================================================
# ROOT DIRECTORIES
# ============================================================

ROOT = Path("/kaggle/input/datasets")

ANISHA = ROOT / "anishapanja"

COAUTHOR = ROOT / "hritishachoudhury"

MANIFEST_ROOT = ANISHA / "bc-xai-final-manifest"
WORK_DIR = Path("/kaggle/working")

WORK_DIR.mkdir(exist_ok=True)
# ============================================================
# DATASET PATHS
# ============================================================

A2_PATH = ANISHA / "bc-xai-a2-2000-partial"

E2_PATH = ANISHA / "bc-xai-e2-2000"

C8_BATCHES = [

    ANISHA / "bc-xai-c8-patches-batch1",

    ANISHA / "bc-xai-c8-patches-batch2",

    ANISHA / "bc-xai-c8-patches-batch3",

    ANISHA / "bc-xai-c8-patches-batch4"

]

A8_BATCHES = [

    ANISHA / "bc-xai-a8-patches-batch1",

    ANISHA / "bc-xai-a8-patches-batch2",

    ANISHA / "bc-xai-a8-patches-batch3",

    ANISHA / "bc-xai-a8-patches-batch4",

    ANISHA / "bc-xai-a8-patches-batch5"

]

D8_PATH = COAUTHOR / "bc-xai-d8-patches"

BH_PATH = COAUTHOR / "bc-xai-bh-patches"
# ============================================================
# VERIFY DATASETS
# ============================================================

ALL_DATASETS = [

    ("A2", A2_PATH),

    ("E2", E2_PATH),

    ("D8", D8_PATH),

    ("BH", BH_PATH)

]

for ds in C8_BATCHES:

    ALL_DATASETS.append((ds.name, ds))

for ds in A8_BATCHES:

    ALL_DATASETS.append((ds.name, ds))

print("=" * 80)
print("VERIFYING ATTACHED KAGGLE DATASETS")
print("=" * 80)

missing = []

for name, path in ALL_DATASETS:

    exists = path.exists()

    print(f"{name:<35} {exists}")

    if not exists:

        missing.append(name)

if len(missing):

    raise FileNotFoundError(
        f"\nMissing Kaggle datasets:\n{missing}"
    )

print("\nAll datasets found successfully.")

# ============================================================
# DEVICE
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("\nDevice :", DEVICE)

if DEVICE.type == "cuda":

    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 80)
# ============================================================
# REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int = 42):

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False

    torch.use_deterministic_algorithms(False)


seed_everything(SEED)

print("=" * 80)
print("Random Seed Fixed")
print("=" * 80)

print(f"Seed : {SEED}")

# ============================================================
# GPU INFORMATION
# ============================================================

print("\n")
print("=" * 80)
print("SYSTEM INFORMATION")
print("=" * 80)

print(f"PyTorch Version : {torch.__version__}")

print(f"CUDA Available : {torch.cuda.is_available()}")

print(f"Device : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU : {torch.cuda.get_device_name(0)}")

    print(
        f"CUDA Version : {torch.version.cuda}"
    )

    gpu_properties = torch.cuda.get_device_properties(0)

    print(
        f"Total GPU Memory : {gpu_properties.total_memory / 1024**3:.2f} GB"
    )

print("=" * 80)
# ============================================================
# LOAD FINAL MANIFEST
# ============================================================

print("=" * 80)
print("LOADING FINAL MANIFEST")
print("=" * 80)

MANIFEST_ROOT = ANISHA / "bc-xai-final-manifest"

PATCH_CSV = MANIFEST_ROOT / "patches_with_split_final.csv"

PATIENT_CSV = MANIFEST_ROOT / "patient_summary.csv"

if not PATCH_CSV.exists():
    raise FileNotFoundError(f"Cannot find:\n{PATCH_CSV}")

if not PATIENT_CSV.exists():
    raise FileNotFoundError(f"Cannot find:\n{PATIENT_CSV}")

patch_df = pd.read_csv(PATCH_CSV)

patch_df["subtype_clean"] = (
    patch_df["subtype_clean"]
    .astype(str)
    .str.strip()
    .replace({
        "Her2": "HER2"
    })
)


patch_df["dataset"] = (
    patch_df["dataset"]
    .astype(str)
    .str.strip()
)

patient_df = pd.read_csv(PATIENT_CSV)

print(f"Patch CSV Loaded    : {PATCH_CSV.name}")
print(f"Patient CSV Loaded  : {PATIENT_CSV.name}")

print()

print("Patch DataFrame Shape :", patch_df.shape)

print("Patient DataFrame Shape :", patient_df.shape)

# ============================================================
# VERIFY REQUIRED COLUMNS
# ============================================================

required_columns = [

    "patient_id",

    "site",

    "subtype",

    "subtype_clean",

    "dataset",

    "relative_path",

    "patch_name",

    "split"

]

missing_columns = [

    col

    for col in required_columns

    if col not in patch_df.columns

]

if len(missing_columns):

    raise ValueError(

        f"Missing Columns:\n{missing_columns}"

    )

print("\nAll required columns present.")

# ============================================================
# LABEL ENCODING
# ============================================================

patch_df["subtype_clean"] = (
    patch_df["subtype_clean"]
    .astype(str)
    .str.strip()
)

CLASS_TO_INDEX = {
    "Basal": 0,
    "HER2": 1,
    "LumA": 2,
    "LumB": 3
}

INDEX_TO_CLASS = {
    v: k
    for k, v in CLASS_TO_INDEX.items()
}

print("\nUnique subtype labels in manifest:")
print(sorted(patch_df["subtype_clean"].unique()))

patch_df["label"] = patch_df["subtype_clean"].map(CLASS_TO_INDEX)

unknown_mask = patch_df["label"].isna()

if unknown_mask.any():

    print("\nUnknown subtype labels found:")
    print(
        patch_df.loc[
            unknown_mask,
            "subtype_clean"
        ].value_counts(dropna=False)
    )

    raise ValueError("Unknown subtype labels detected.")

patch_df["label"] = patch_df["label"].astype(int)
# ============================================================
# CLASS NAMES (ORDER MATCHES LABEL INDICES)
# ============================================================

CLASS_NAMES = [

    INDEX_TO_CLASS[i]

    for i in range(len(INDEX_TO_CLASS))

]

print("Class Mapping:")

for idx, class_name in enumerate(CLASS_NAMES):

    print(f"{idx}: {class_name}")
# ============================================================
# TRAIN / VALIDATION / TEST
# ============================================================

train_df = (

    patch_df

    [patch_df["split"]=="train"]

    .reset_index(drop=True)

)

val_df = (

    patch_df

    [patch_df["split"]=="val"]

    .reset_index(drop=True)

)

test_df = (

    patch_df

    [patch_df["split"]=="test"]

    .reset_index(drop=True)

)

# ============================================================
# SUMMARY
# ============================================================

print("\n")

print("=" * 80)

print("DATASET SUMMARY")

print("=" * 80)

print(f"Training Patches   : {len(train_df):,}")

print(f"Validation Patches : {len(val_df):,}")

print(f"Testing Patches    : {len(test_df):,}")

print()

print("Training Class Distribution")

print(

    train_df["subtype_clean"]

    .value_counts()

    .sort_index()

)

print()

print("Validation Class Distribution")

print(

    val_df["subtype_clean"]

    .value_counts()

    .sort_index()

)

print()

print("Testing Class Distribution")

print(

    test_df["subtype_clean"]

    .value_counts()

    .sort_index()

)

print()

print("=" * 80)

print("Manifest Loaded Successfully")

print("=" * 80)
# ============================================================
# IMAGE TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.80, 1.00),
        ratio=(0.90, 1.10)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomVerticalFlip(p=0.5),

    transforms.RandomRotation(degrees=15),

    transforms.ColorJitter(
        brightness=0.20,
        contrast=0.20,
        saturation=0.15,
        hue=0.02
    ),

    transforms.RandomAutocontrast(p=0.20),

    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 1.5)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])


val_transform = transforms.Compose([

    transforms.Resize(256),

    transforms.CenterCrop(IMAGE_SIZE),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

test_transform = val_transform


# ============================================================
# DATASET ROOT MAPPING
# ============================================================

DATASET_ROOTS = {

    "bc-xai-a2-2000-partial": A2_PATH,

    "bc-xai-e2-2000": E2_PATH,

    "bc-xai-d8-patches": D8_PATH,

    "bc-xai-bh-patches": BH_PATH,

    "bc-xai-c8-patches-batch1": C8_BATCHES[0],
    "bc-xai-c8-patches-batch2": C8_BATCHES[1],
    "bc-xai-c8-patches-batch3": C8_BATCHES[2],
    "bc-xai-c8-patches-batch4": C8_BATCHES[3],

    "bc-xai-a8-patches-batch1": A8_BATCHES[0],
    "bc-xai-a8-patches-batch2": A8_BATCHES[1],
    "bc-xai-a8-patches-batch3": A8_BATCHES[2],
    "bc-xai-a8-patches-batch4": A8_BATCHES[3],
    "bc-xai-a8-patches-batch5": A8_BATCHES[4]

}


# ============================================================
# CUSTOM DATASET
# ============================================================

class BreastCancerDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(self, index):

        row = self.df.iloc[index]

        dataset_name = str(row["dataset"]).strip()

        relative_path = Path(str(row["relative_path"]).strip())

        if dataset_name not in DATASET_ROOTS:

            raise ValueError(
                f"Unknown dataset '{dataset_name}'."
            )

        image_path = DATASET_ROOTS[dataset_name] / relative_path

        if not image_path.exists():

            raise FileNotFoundError(

                f"""
Image not found.

Dataset      : {dataset_name}

Relative Path: {relative_path}

Full Path    : {image_path}
"""
            )

        image = Image.open(image_path).convert("RGB")

        if self.transform is not None:

            image = self.transform(image)

        label = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        sample = {

            "image": image,

            "label": label,

            "patient_id": row["patient_id"],

            "site": row["site"],

            "subtype": row["subtype_clean"],

            "dataset": dataset_name,

            "path": str(image_path)

        }

        return sample


# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = BreastCancerDataset(

    dataframe=train_df,

    transform=train_transform

)

val_dataset = BreastCancerDataset(

    dataframe=val_df,

    transform=val_transform

)

test_dataset = BreastCancerDataset(

    dataframe=test_df,

    transform=test_transform

)


# ============================================================
# DATASET SUMMARY
# ============================================================

print("=" * 80)

print("DATASETS CREATED")

print("=" * 80)

print(f"Training Samples   : {len(train_dataset):,}")

print(f"Validation Samples : {len(val_dataset):,}")

print(f"Testing Samples    : {len(test_dataset):,}")

print("=" * 80)
# ============================================================
# COMPUTE CLASS WEIGHTS
# ============================================================

print("=" * 80)
print("COMPUTING CLASS WEIGHTS")
print("=" * 80)

train_labels = train_df["label"].values

class_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES
)

class_weights = 1.0 / class_counts

sample_weights = class_weights[train_labels]

sample_weights = torch.DoubleTensor(sample_weights)

weighted_sampler = WeightedRandomSampler(

    weights=sample_weights,

    num_samples=len(sample_weights),

    replacement=True

)

print("Training Class Counts")

for idx, count in enumerate(class_counts):

    print(f"{INDEX_TO_CLASS[idx]:<6} : {count:,}")

print()

print("Class Weights")

for idx, weight in enumerate(class_weights):

    print(f"{INDEX_TO_CLASS[idx]:<6} : {weight:.6f}")

print("=" * 80)


# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(

    dataset=train_dataset,

    batch_size=BATCH_SIZE,

    sampler=weighted_sampler,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=NUM_WORKERS > 0,

    drop_last=True

)


val_loader = DataLoader(

    dataset=val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=NUM_WORKERS > 0

)


test_loader = DataLoader(

    dataset=test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    pin_memory=True,

    persistent_workers=NUM_WORKERS > 0

)


# ============================================================
# VERIFY DATALOADERS
# ============================================================

print("\n")

print("=" * 80)
print("DATALOADER SUMMARY")
print("=" * 80)

print(f"Training Batches   : {len(train_loader):,}")

print(f"Validation Batches : {len(val_loader):,}")

print(f"Testing Batches    : {len(test_loader):,}")

print("=" * 80)

# ============================================================
# BUILD SWIN-T MODEL
# ============================================================

print("=" * 80)
print("BUILDING SWIN-T")
print("=" * 80)

model = swin_t(
    weights=Swin_T_Weights.IMAGENET1K_V1
)

num_features = model.head.in_features

model.head = nn.Linear(
    in_features=num_features,
    out_features=NUM_CLASSES
)

model = model.to(DEVICE)

print(model)
print()

print(f"Feature Dimension : {num_features}")
print(f"Number of Classes : {NUM_CLASSES}")

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")

print("=" * 80)


# ============================================================
# VERIFY MODEL
# ============================================================

dummy_input = torch.randn(
    2,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE
)

with torch.no_grad():
    dummy_output = model(dummy_input)

print()
print("Model Verification")
print(f"Input Shape  : {dummy_input.shape}")
print(f"Output Shape : {dummy_output.shape}")

assert dummy_output.shape == (2, NUM_CLASSES)

print("Forward Pass Successful")
print("=" * 80)


# ============================================================
# LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=LABEL_SMOOTHING
)


# ============================================================
# OPTIMIZER
# ============================================================

decay = []
no_decay = []

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.ndim == 1 or name.endswith(".bias"):
        no_decay.append(param)
    else:
        decay.append(param)

optimizer = optim.AdamW(
    [
        {
            "params": decay,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": no_decay,
            "weight_decay": 0.0
        }
    ],
    lr=LEARNING_RATE
)


# ============================================================
# LEARNING RATE SCHEDULER
# ============================================================

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer=optimizer,
    T_0=10,
    T_mult=2,
    eta_min=1e-6
)


# ============================================================
# AUTOMATIC MIXED PRECISION (AMP)
# ============================================================

scaler = GradScaler(
    enabled=torch.cuda.is_available()
)


# ============================================================
# TRAINING STATE — FRESH RUN
# ============================================================

best_val_accuracy = 0.0
best_val_f1 = 0.0
best_epoch = -1

epochs_without_improvement = 0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_f1": [],
    "learning_rate": []
}


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

print("Model                : Swin-T")
print(f"Image Size           : {IMAGE_SIZE}")
print(f"Number of Classes    : {NUM_CLASSES}")

print()

print("Loss Function        : CrossEntropyLoss")
print(f"Label Smoothing      : {LABEL_SMOOTHING}")

print()

print("Optimizer            : AdamW")
print(f"Learning Rate        : {LEARNING_RATE}")
print(f"Weight Decay         : {WEIGHT_DECAY}")

print()

print("Scheduler            : CosineAnnealingWarmRestarts")
print("T_0                  : 10")
print("T_mult               : 2")
print("Minimum LR           : 1e-6")

print()

print(f"Mixed Precision      : {torch.cuda.is_available()}")

print()

print(f"Early Stopping       : {EARLY_STOPPING_PATIENCE} epochs")

print("=" * 80)

# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    scheduler,
    scaler,
    device,
    epoch
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    progress_bar = tqdm(
        dataloader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]",
        leave=False
    )

    for batch in progress_bar:

        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRADIENT_CLIP
        )

        scaler.step(optimizer)
        scaler.update()

        predictions = outputs.argmax(dim=1)
        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size
        running_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        current_loss = running_loss / total_samples
        current_accuracy = 100.0 * running_correct / total_samples
        current_lr = optimizer.param_groups[0]["lr"]

        progress_bar.set_postfix({
            "Loss": f"{current_loss:.4f}",
            "Acc": f"{current_accuracy:.2f}%",
            "LR": f"{current_lr:.2e}"
        })

    scheduler.step(epoch + 1)

    epoch_loss = running_loss / total_samples
    epoch_accuracy = running_correct / total_samples

    return {
        "loss": epoch_loss,
        "accuracy": epoch_accuracy,
        "learning_rate": optimizer.param_groups[0]["lr"]
    }


# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate_one_epoch(
    model,
    dataloader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    total_samples = 0

    all_labels = []
    all_predictions = []
    all_probabilities = []

    progress_bar = tqdm(
        dataloader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():

        for batch in progress_bar:

            images = batch["image"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)

            probabilities = torch.softmax(outputs, dim=1)
            predictions = torch.argmax(outputs, dim=1)

            batch_size = labels.size(0)

            running_loss += loss.item() * batch_size
            running_correct += (predictions == labels).sum().item()
            total_samples += batch_size

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

            progress_bar.set_postfix({
                "Loss": f"{running_loss / total_samples:.4f}",
                "Acc": (
                    f"{100.0 * running_correct / total_samples:.2f}%"
                )
            })

    epoch_loss = running_loss / total_samples
    epoch_accuracy = running_correct / total_samples

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_predictions
    )

    return {
        "loss": epoch_loss,
        "accuracy": epoch_accuracy,
        "macro_f1": macro_f1,
        "labels": np.array(all_labels),
        "predictions": np.array(all_predictions),
        "probabilities": np.array(all_probabilities),
        "balanced_accuracy": balanced_accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall
    }


# ============================================================
# MODEL TRAINING — SWIN-T (RESUMABLE)
# Matches the ResNet50 training structure
# ============================================================

import time
import shutil
from pathlib import Path
import numpy as np
import torch

BEST_MODEL_PATH = WORK_DIR / "best_swin_t.pth"
LAST_MODEL_PATH = WORK_DIR / "last_swin_t.pth"

# ------------------------------------------------------------
# FIND CHECKPOINTS IN ATTACHED KAGGLE DATASETS
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/input")
last_candidates = sorted(INPUT_ROOT.rglob("last_swin_t.pth"))

print("=" * 80)
print("STARTING SWIN-T TRAINING")
print("=" * 80)

for candidate in last_candidates:
    print("Available last checkpoint:", candidate)

# ------------------------------------------------------------
# INITIALIZE FRESH-RUN DEFAULTS
# ------------------------------------------------------------

start_epoch = 0
best_val_accuracy = 0.0
best_val_f1 = -float("inf")
best_epoch = -1
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_f1": [],
    "learning_rate": [],
}

# ------------------------------------------------------------
# RESUME TRAINING IF AN ATTACHED CHECKPOINT EXISTS
# ------------------------------------------------------------

if last_candidates:

    # If several checkpoints exist, select the first sorted path.
    # For unambiguous recovery, attach only the intended run.
    RESUME_LAST_MODEL = last_candidates[0]

    print("\n" + "=" * 80)
    print("RESUMING SWIN-T TRAINING")
    print("=" * 80)
    print("Checkpoint:", RESUME_LAST_MODEL)

    checkpoint = torch.load(
        RESUME_LAST_MODEL,
        map_location=DEVICE,
        weights_only=False,
    )

    required_keys = [
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "best_val_accuracy",
        "best_val_f1",
        "best_epoch",
        "epochs_without_improvement",
        "history",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in checkpoint
    ]

    if missing_keys:
        raise KeyError(
            f"Checkpoint is missing required keys: {missing_keys}"
        )

    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    else:
        print("Warning: no AMP scaler state; using a fresh scaler.")

    start_epoch = int(checkpoint["epoch"])
    best_val_accuracy = checkpoint["best_val_accuracy"]
    best_val_f1 = checkpoint["best_val_f1"]
    best_epoch = checkpoint["best_epoch"]
    epochs_without_improvement = checkpoint[
        "epochs_without_improvement"
    ]
    history = checkpoint["history"]

    # Recover the matching best checkpoint from the same dataset
    # directory as the selected last checkpoint.
    attached_best = RESUME_LAST_MODEL.parent / "best_swin_t.pth"

    if attached_best.exists():
        shutil.copy2(attached_best, BEST_MODEL_PATH)
    elif not BEST_MODEL_PATH.exists():
        raise FileNotFoundError(
            "The attached dataset has last_swin_t.pth but no matching "
            "best_swin_t.pth. Preserve both checkpoints together."
        )

    print(f"Completed epochs: {start_epoch}")
    print(f"Next epoch: {start_epoch + 1}")
    print(f"Best epoch so far: {best_epoch}")
    print(f"Best validation Macro F1: {best_val_f1:.4f}")

else:
    print("\nNo attached checkpoint found. Starting from scratch.")

# ------------------------------------------------------------
# TRAINING LOOP
# ------------------------------------------------------------

for epoch in range(start_epoch, EPOCHS):

    epoch_start = time.time()

    train_results = train_one_epoch(
        model=model,
        dataloader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=DEVICE,
        epoch=epoch,
    )

    val_results = validate_one_epoch(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )

    epoch_time = time.time() - epoch_start

    # Update history
    history["train_loss"].append(train_results["loss"])
    history["train_accuracy"].append(train_results["accuracy"])
    history["val_loss"].append(val_results["loss"])
    history["val_accuracy"].append(val_results["accuracy"])
    history["val_f1"].append(val_results["macro_f1"])
    history["learning_rate"].append(train_results["learning_rate"])

    # Best model: Macro F1, with accuracy as tie-breaker
    improved = (
        val_results["macro_f1"] > best_val_f1
        or (
            np.isclose(val_results["macro_f1"], best_val_f1)
            and val_results["accuracy"] > best_val_accuracy
        )
    )

    if improved:
        best_val_f1 = val_results["macro_f1"]
        best_val_accuracy = val_results["accuracy"]
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        status = "Improved"
    else:
        epochs_without_improvement += 1
        status = "No improvement"

    # Full state needed for resuming
    checkpoint_state = {
        "model_name": "Swin-T",
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_accuracy": best_val_accuracy,
        "best_val_f1": best_val_f1,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "history": history,
    }

    # Save the latest checkpoint after every completed epoch
    torch.save(checkpoint_state, LAST_MODEL_PATH)

    # Save best checkpoint only when validation performance improves
    if improved:
        torch.save(checkpoint_state, BEST_MODEL_PATH)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] | "
        f"Train Loss: {train_results['loss']:.4f} | "
        f"Train Acc: {train_results['accuracy']:.4f} | "
        f"Val Loss: {val_results['loss']:.4f} | "
        f"Val Acc: {val_results['accuracy']:.4f} | "
        f"Val Macro F1: {val_results['macro_f1']:.4f} | "
        f"Time: {epoch_time / 60:.2f} min | {status}"
    )

    print(f"Best Val Macro F1: {best_val_f1:.4f} (Epoch {best_epoch})")
    print(f"Latest checkpoint saved: {LAST_MODEL_PATH}")

    if improved:
        print(f"Best checkpoint saved: {BEST_MODEL_PATH}")

    print("-" * 80)

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping triggered at epoch {epoch + 1}.")
        break

print("=" * 80)
print("SWIN-T TRAINING FINISHED")
print(f"Best epoch: {best_epoch}")
print(f"Best validation Macro F1: {best_val_f1:.4f}")
print(f"Best checkpoint exists: {BEST_MODEL_PATH.exists()}")
print(f"Latest checkpoint exists: {LAST_MODEL_PATH.exists()}")
print("=" * 80)
# ============================================================
# SWIN-T: PACKAGE CHECKPOINTS FOR SESSION RECOVERY
# ============================================================

import zipfile

checkpoint_archive = WORK_DIR / "swin_t_resume_checkpoints.zip"

with zipfile.ZipFile(
    checkpoint_archive,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for checkpoint_path in [LAST_MODEL_PATH, BEST_MODEL_PATH]:
        if checkpoint_path.exists():
            archive.write(
                checkpoint_path,
                arcname=checkpoint_path.name,
            )

print("Checkpoint archive:", checkpoint_archive)
print("Latest checkpoint exists:", LAST_MODEL_PATH.exists())
print("Best checkpoint exists:", BEST_MODEL_PATH.exists())
# ============================================================
# SWIN-T: TRAINING HISTORY VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

epochs_completed = len(history["train_loss"])

if epochs_completed == 0:
    raise RuntimeError("No completed training epochs are available.")

epochs = range(1, epochs_completed + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
axes[0, 0].plot(epochs, history["train_loss"], label="Train")
axes[0, 0].plot(epochs, history["val_loss"], label="Validation")
axes[0, 0].set_title("Swin-T Loss")
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Cross-Entropy Loss")
axes[0, 0].grid(True)
axes[0, 0].legend()

# Accuracy
axes[0, 1].plot(
    epochs, np.array(history["train_accuracy"]) * 100,
    label="Train"
)
axes[0, 1].plot(
    epochs, np.array(history["val_accuracy"]) * 100,
    label="Validation"
)
axes[0, 1].set_title("Swin-T Accuracy")
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy (%)")
axes[0, 1].grid(True)
axes[0, 1].legend()

# Validation Macro F1
axes[1, 0].plot(epochs, history["val_f1"])
axes[1, 0].set_title("Swin-T Validation Macro F1")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Macro F1")
axes[1, 0].grid(True)

# Learning rate
axes[1, 1].plot(epochs, history["learning_rate"])
axes[1, 1].set_title("Swin-T Learning Rate Schedule")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Learning Rate")
axes[1, 1].set_yscale("log")
axes[1, 1].grid(True)

plt.tight_layout()

history_plot_path = WORK_DIR / "swin_t_training_history.png"
plt.savefig(history_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Training history saved: {history_plot_path}")
# ============================================================
# SWIN-T: HELD-OUT TEST SET EVALUATION
# ============================================================

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found: {BEST_MODEL_PATH}"
    )

best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(best_checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

best_epoch = best_checkpoint["best_epoch"]
best_val_accuracy = best_checkpoint["best_val_accuracy"]
best_val_f1 = best_checkpoint["best_val_f1"]

print(f"Evaluating best epoch: {best_epoch}")
print(f"Best validation Macro F1: {best_val_f1:.4f}")

test_results = validate_one_epoch(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=DEVICE,
)

# Extract the dictionary values for the remaining evaluation cells.
test_loss = test_results["loss"]
test_accuracy = test_results["accuracy"]
test_labels = np.asarray(test_results["labels"])
test_predictions = np.asarray(test_results["predictions"])
test_probabilities = np.asarray(test_results["probabilities"])
test_macro_f1 = test_results["macro_f1"]
test_balanced_accuracy = test_results["balanced_accuracy"]
test_macro_precision = test_results["macro_precision"]
test_macro_recall = test_results["macro_recall"]

print("\n" + "=" * 80)
print("SWIN-T — HELD-OUT TEST RESULTS")
print("=" * 80)
print(f"Test Loss:              {test_loss:.4f}")
print(f"Test Accuracy:          {test_accuracy * 100:.2f}%")
print(f"Balanced Accuracy:      {test_balanced_accuracy:.4f}")
print(f"Macro Precision:        {test_macro_precision:.4f}")
print(f"Macro Recall:           {test_macro_recall:.4f}")
print(f"Macro F1-score:         {test_macro_f1:.4f}")
print("=" * 80)

report = classification_report(
    test_labels,
    test_predictions,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
    output_dict=True,
)

report_df = pd.DataFrame(report).transpose()
report_path = WORK_DIR / "swin_t_classification_report.csv"
report_df.to_csv(report_path, index=True)

print("\nClassification Report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
)
print(f"Classification report saved: {report_path}")
# ============================================================
# SWIN-T: CONFUSION MATRIX
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=list(range(NUM_CLASSES)),
)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)

plt.title("Swin-T Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()

cm_path = WORK_DIR / "swin_t_confusion_matrix.png"
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Confusion matrix saved to: {cm_path}")
# ============================================================
# SWIN-T: ROC CURVES
# ============================================================

import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_binary = label_binarize(
    test_labels,
    classes=list(range(NUM_CLASSES)),
)

if test_probabilities.shape != (len(test_labels), NUM_CLASSES):
    raise ValueError(
        f"Unexpected probability shape: {test_probabilities.shape}. "
        f"Expected ({len(test_labels)}, {NUM_CLASSES})."
    )

plt.figure(figsize=(9, 7))

roc_results = {}

for class_idx, class_name in enumerate(CLASS_NAMES):
    # ROC is undefined if the test set has no positive or no negative
    # examples for a class.
    if len(np.unique(y_test_binary[:, class_idx])) < 2:
        print(f"Skipping ROC for {class_name}: only one binary label present.")
        continue

    fpr, tpr, _ = roc_curve(
        y_test_binary[:, class_idx],
        test_probabilities[:, class_idx],
    )
    roc_auc = auc(fpr, tpr)
    roc_results[class_name] = roc_auc

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {roc_auc:.4f})",
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.title("Swin-T One-vs-Rest ROC Curves")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()

roc_path = WORK_DIR / "swin_t_roc_curve.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.show()

print("ROC AUC by class:")
for class_name, roc_auc in roc_results.items():
    print(f"{class_name}: {roc_auc:.4f}")

print(f"ROC curve saved to: {roc_path}")
# ============================================================
# SWIN-T: PRECISION-RECALL CURVES
# ============================================================

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(9, 7))

ap_results = {}

for class_idx, class_name in enumerate(CLASS_NAMES):
    y_true_binary = y_test_binary[:, class_idx]

    if y_true_binary.sum() == 0:
        print(
            f"Skipping precision-recall curve for {class_name}: "
            "no positive test examples."
        )
        continue

    precision_values, recall_values, _ = precision_recall_curve(
        y_true_binary,
        test_probabilities[:, class_idx],
    )

    average_precision = average_precision_score(
        y_true_binary,
        test_probabilities[:, class_idx],
    )
    ap_results[class_name] = average_precision

    plt.plot(
        recall_values,
        precision_values,
        label=f"{class_name} (AP = {average_precision:.4f})",
    )

plt.title("Swin-T One-vs-Rest Precision-Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="best")
plt.grid(True, alpha=0.3)
plt.tight_layout()

pr_path = WORK_DIR / "swin_t_precision_recall_curve.png"
plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.show()

print("Average precision by class:")
for class_name, average_precision in ap_results.items():
    print(f"{class_name}: {average_precision:.4f}")

print(f"Precision-recall curves saved to: {pr_path}")
# ============================================================
# SWIN-T: EXPORT TEST PREDICTIONS
# ============================================================

if len(test_df) != len(test_labels):
    raise ValueError(
        f"test_df has {len(test_df)} rows but evaluation returned "
        f"{len(test_labels)} labels. Check test_loader ordering and dataset."
    )

test_confidence = np.max(test_probabilities, axis=1)

predictions_df = test_df.reset_index(drop=True).copy()

predictions_df["True_Label"] = test_labels
predictions_df["Predicted_Label"] = test_predictions
predictions_df["True_Class"] = [
    CLASS_NAMES[int(label)] for label in test_labels
]
predictions_df["Predicted_Class"] = [
    CLASS_NAMES[int(label)] for label in test_predictions
]
predictions_df["Confidence"] = test_confidence

for class_idx, class_name in enumerate(CLASS_NAMES):
    predictions_df[f"Prob_{class_name}"] = test_probabilities[:, class_idx]

predictions_path = WORK_DIR / "swin_t_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

print(f"Test predictions saved to: {predictions_path}")
print(f"Number of predictions: {len(predictions_df):,}")
display(predictions_df.head())
# ============================================================
# SWIN-T: MISCLASSIFICATION ANALYSIS
# ============================================================

predictions_df["Correct"] = (
    predictions_df["True_Label"] == predictions_df["Predicted_Label"]
)

correct_df = predictions_df[predictions_df["Correct"]].copy()
incorrect_df = predictions_df[~predictions_df["Correct"]].copy()

print(f"Correct predictions:      {len(correct_df):,}")
print(f"Incorrect predictions:    {len(incorrect_df):,}")
print(
    f"Observed test error rate: "
    f"{len(incorrect_df) / len(predictions_df):.4%}"
)

print("\nTop 20 highest-confidence incorrect predictions:")
if not incorrect_df.empty:
    display(
        incorrect_df.sort_values(
            "Confidence", ascending=False
        ).head(20)
    )
else:
    print("No incorrect predictions.")

print("\nTop 20 lowest-confidence correct predictions:")
if not correct_df.empty:
    display(
        correct_df.sort_values(
            "Confidence", ascending=True
        ).head(20)
    )
else:
    print("No correct predictions.")
# ============================================================
# SWIN-T: FINAL EXPERIMENT SUMMARY
# ============================================================

summary = {
    "Architecture": "Swin-T",
    "Best Epoch": best_epoch,
    "Validation Accuracy": best_val_accuracy,
    "Validation Macro F1": best_val_f1,
    "Test Loss": test_loss,
    "Test Accuracy": accuracy_score(test_labels, test_predictions),
    "Balanced Accuracy": balanced_accuracy_score(
        test_labels, test_predictions
    ),
    "Macro Precision": precision_score(
        test_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0,
    ),
    "Macro Recall": recall_score(
        test_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0,
    ),
    "Macro F1": f1_score(
        test_labels,
        test_predictions,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0,
    ),
    "Test Samples": len(test_labels),
}

summary_df = pd.DataFrame([summary])
summary_path = WORK_DIR / "swin_t_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\nFinal Swin-T Summary:")
display(summary_df)

print("\nAll evaluation outputs have been saved.")
print(f"Output directory: {WORK_DIR}")
print(f" - {history_plot_path.name}")
print(f" - {report_path.name}")
print(f" - {cm_path.name}")
print(f" - {roc_path.name}")
print(f" - {pr_path.name}")
print(f" - {predictions_path.name}")
print(f" - {summary_path.name}")